# Compound Weather Event Analysis
Visualizes co-occurrence patterns and interaction effects between weather event categories.
Generates Figure 4 for the research paper.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from itertools import combinations

from src.features.compound_events import (
    CompoundEventFeatureBuilder, classify_event,
    EVENT_CATEGORIES, CATEGORY_PAIRS
)

plt.style.use('dark_background')
print('Compound Event Analysis')

In [ ]:
# Load storm events and classify
storms = pd.read_csv('../data/raw/storm_events_details.csv', low_memory=False)
storms.columns = [c.upper() for c in storms.columns]
storms['BEGIN_DATE_TIME'] = pd.to_datetime(storms['BEGIN_DATE_TIME'], format='mixed', errors='coerce')
storms = storms.dropna(subset=['BEGIN_DATE_TIME'])
storms['CATEGORY'] = storms['EVENT_TYPE'].apply(classify_event)
classified = storms.dropna(subset=['CATEGORY'])
print(f'{len(classified):,} events classified into {classified["CATEGORY"].nunique()} categories')

In [ ]:
# Build co-occurrence matrix
# For each day, check which categories are active
classified['date'] = classified['BEGIN_DATE_TIME'].dt.date
categories = sorted(EVENT_CATEGORIES.keys())

cooccurrence = np.zeros((len(categories), len(categories)))

for date, group in classified.groupby('date'):
    active = set(group['CATEGORY'].unique())
    for i, cat_a in enumerate(categories):
        for j, cat_b in enumerate(categories):
            if cat_a in active and cat_b in active:
                cooccurrence[i][j] += 1

# Normalize by total days
total_days = classified['date'].nunique()
cooccurrence_pct = cooccurrence / total_days * 100

# Plot heatmap (Paper Figure 4)
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(cooccurrence_pct, dtype=bool), k=1)
sns.heatmap(
    cooccurrence_pct, mask=mask, annot=True, fmt='.1f',
    xticklabels=categories, yticklabels=categories,
    cmap='YlOrRd', ax=ax, cbar_kws={'label': '% of Days'}
)
ax.set_title('Weather Event Category Co-Occurrence (% of Days)')
plt.tight_layout()
plt.savefig('../paper/figures/compound_cooccurrence_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compound events vs outage severity
# Shows that multi-category days have higher average damage
daily_stats = []
for date, group in classified.groupby('date'):
    n_categories = group['CATEGORY'].nunique()
    total_damage = pd.to_numeric(group.get('DAMAGE_PROPERTY', pd.Series(0)), errors='coerce').sum()
    daily_stats.append({'date': date, 'n_categories': n_categories, 'total_damage': total_damage})

daily_df = pd.DataFrame(daily_stats)

fig, ax = plt.subplots(figsize=(8, 5))
grouped = daily_df.groupby('n_categories')['total_damage'].mean()
grouped.plot(kind='bar', ax=ax, color=['#22c55e', '#eab308', '#f97316', '#ef4444', '#dc2626', '#991b1b'][:len(grouped)])
ax.set_xlabel('Number of Active Weather Categories')
ax.set_ylabel('Average Daily Property Damage ($)')
ax.set_title('Property Damage vs Compound Event Complexity')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../paper/figures/compound_vs_damage.png', dpi=300, bbox_inches='tight')
plt.show()